In [ ]:
%%writefile submission.py
def agent(obs, config):
    import random

    if not hasattr(obs, 'board') or obs.board is None:
        return 0

    board = obs.board
    columns = config.columns
    rows = config.rows
    inarow = config.inarow
    
    mark = obs.mark
    opponent_mark = 2 if mark == 1 else 1

    def drop_piece(grid, col, current_mark):
        for r in range(rows - 1, -1, -1):
            if grid[r * columns + col] == 0:
                next_grid = list(grid)
                next_grid[r * columns + col] = current_mark
                return next_grid
        return None

    def evaluate_window(window, current_mark):
        score = 0
        opp_mark = 2 if current_mark == 1 else 1
        
        if window.count(current_mark) == inarow:
            return 1000000
        elif window.count(current_mark) == inarow - 1 and window.count(0) == 1:
            score += 1000
        elif window.count(current_mark) == inarow - 2 and window.count(0) == 2:
            score += 10
            
        if window.count(opp_mark) == inarow - 1 and window.count(0) == 1:
            score -= 900  # ضغط قاسي لمنع الخصم
            
        return score

    def score_board(grid, current_mark):
        score = 0
        center = columns // 2
        center_array = [grid[r * columns + center] for r in range(rows)]
        score += center_array.count(current_mark) * 20
        
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
        for r in range(rows):
            for c in range(columns):
                for dr, dc in directions:
                    window = []
                    for i in range(inarow):
                        nr, nc = r + i * dr, c + i * dc
                        if 0 <= nr < rows and 0 <= nc < columns:
                            window.append(grid[nr * columns + nc])
                        else:
                            break
                    if len(window) == inarow:
                        score += evaluate_window(window, current_mark)
        return score

    def minimax(grid, depth, maximizing_player, alpha, beta):
        valid_actions = [c for c in range(columns) if grid[c] == 0]
        
        is_terminal = len(valid_actions) == 0 or score_board(grid, mark) >= 1000000 or score_board(grid, opponent_mark) >= 1000000
        if depth == 0 or is_terminal:
            return score_board(grid, mark) - score_board(grid, opponent_mark)

        if maximizing_player:
            max_eval = -float('inf')
            for col in valid_actions:
                next_grid = drop_piece(grid, col, mark)
                if next_grid:
                    eval = minimax(next_grid, depth - 1, False, alpha, beta)
                    max_eval = max(max_eval, eval)
                    alpha = max(alpha, eval)
                    if beta <= alpha:
                        break
            return max_eval
        else:
            min_eval = float('inf')
            for col in valid_actions:
                next_grid = drop_piece(grid, col, opponent_mark)
                if next_grid:
                    eval = minimax(next_grid, depth - 1, True, alpha, beta)
                    min_eval = min(min_eval, eval)
                    beta = min(beta, eval)
                    if beta <= alpha:
                        break
            return min_eval

    valid_actions = [col for col in range(columns) if board[col] == 0]
    if not valid_actions:
        return 0

    # 1. الفوز المباشر الفوري
    for col in valid_actions:
        next_grid = drop_piece(board, col, mark)
        if next_grid and score_board(next_grid, mark) >= 1000000:
            return col

    # 2. منع الخصم من الفوز الفوري
    for col in valid_actions:
        next_grid = drop_piece(board, col, opponent_mark)
        if next_grid and score_board(next_grid, opponent_mark) >= 1000000:
            return col

    # 3. اختيار أفضل حركة باستخدام Minimax مع Alpha-Beta Pruning بعمق 3
    best_score = -float('inf')
    best_col = valid_actions[0]
    
    center = columns // 2
    valid_actions.sort(key=lambda x: abs(x - center))

    for col in valid_actions:
        next_grid = drop_piece(board, col, mark)
        if not next_grid:
            continue
        
        score = minimax(next_grid, 3, False, -float('inf'), float('inf'))
        
        if score > best_score:
            best_score = score
            best_col = col

    return best_col
    